In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, make_scorer
import spyndex as spx
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, make_scorer
import spyndex as spx
import seaborn as sns
import matplotlib.pyplot as plt

d:\Projects\ds_ml_ai\zindi\challenges\amini-canopy-or-crop-challenge\amini\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import pandas as pd
from amini_canopy.config import *

Preview train and test data

In [ ]:
train_data_path = RAW_DATA_DIR / "Train.csv"
raw_train_df = pd.read_csv(train_data_path)

In [ ]:
raw_train_df.head(5)

Check unique row ids

In [ ]:
unique_row_ids = raw_train_df['ID'].unique()
print(f"Number of unique row ids: {len(unique_row_ids)}")

Check missing values

In [ ]:
missing_values_by_id = raw_train_df.groupby('ID'
                                            ).agg(lambda x: x.isnull().sum())
                                        
missing_values_by_id

In [ ]:
non_values_by_id = raw_train_df.groupby('ID').agg(lambda group: group.notnull().sum())
non_values_by_id.head()

In [ ]:
timeseries = raw_train_df.groupby('ID').agg({'ID': 'count'})
timeseries.head()

In [ ]:
# Ensure the 'time' column is in datetime format
raw_train_df['time'] = pd.to_datetime(raw_train_df['time'], errors='coerce')

# Calculate the time range for each unique ID
time_range_by_id = raw_train_df.groupby('ID').agg(
    time_range=('time', lambda x: (x.max() - x.min()).days if not x.isnull().all() else None)
)

time_range_by_id.head()

In [ ]:
unique_pixel_observations = raw_train_df['ID'].sort_values().value_counts()
unique_pixel_observations.head()

Fill missing values using forward fill

In [ ]:
# Calculate the number of unique pixel observations for each unique ID
corrected_train_data = raw_train_df.bfill()
corrected_train_data.head()

In [ ]:
grouped_data = corrected_train_data.groupby('ID')
new_df = pd.DataFrame()
dfs = []
for group in grouped_data:
    chip_data = grouped_data.get_group(group[0]).bfill().ffill()
    dfs.append(chip_data)
new_df = pd.concat(dfs)
new_df.head()

Feature Engineering

Derive indices for detecting canopy from raw spectral indices

In [ ]:
%%script echo skipping
# Calculate vegetation indices
new_df['CSI'] = (new_df['SWIR1'] - new_df['NIR']) / (new_df['SWIR1'] + new_df['NIR'])  # Canopy/Shadow Index
new_df['BSI'] = ((new_df['SWIR1'] + new_df['RED']) - (new_df['NIR'] + new_df['Blue'])) / \
                ((new_df['SWIR1'] + new_df['RED']) + (new_df['NIR'] + new_df['Blue']))  # Bare Soil Index
new_df['AVI'] = (new_df['NIR'] * (1 - new_df['RED']) * (new_df['NIR'] - new_df['RED'])) ** 0.333  # Advanced Vegetation Index

# Display the first few rows of the dataframe with the new indices
new_df[['ID', 'time', 'NDVI', 'LAI', 'CSI', 'BSI', 'AVI']].head()

In [ ]:
test_data_path = RAW_DATA_DIR / "Test.csv"
raw_test_data = pd.read_csv(test_data_path)
raw_test_data.head()

In [ ]:
new_df.ID.nunique(), raw_test_data.ID.nunique()

In [ ]:
dict([(index[0], value) for index, value in new_df.value_counts(subset=['ID']).to_dict().items()])

In [ ]:
max_ts_len = max([(index[0], value) for index, value in new_df.value_counts(subset=['ID']).to_dict().items()], key=lambda x: x[1])
min_ts_len = min([(index[0], value) for index, value in new_df.value_counts(subset=['ID']).to_dict().items()], key=lambda x: x[1])

In [ ]:
max_ts_len, min_ts_len


In [ ]:
max_ts_len_test = max([(index[0], value) for index, value in raw_test_data.value_counts(subset=['ID']).to_dict().items()], key=lambda x: x[1])
min_ts_len_test = min([(index[0], value) for index, value in raw_test_data.value_counts(subset=['ID']).to_dict().items()], key=lambda x: x[1])
max_ts_len_test, min_ts_len_test

In [ ]:
def sample_ts_data(data, n_samples=69):
    all_data = []
    data_grouped = data.groupby('ID')
    for group in data_grouped:
        chip_data = data_grouped.get_group(group[0])
        new_chip_data = chip_data.iloc[-n_samples:]
        all_data.append(new_chip_data)
    return pd.concat(all_data).reset_index(drop=True)

In [ ]:
sampled_test_data = sample_ts_data(raw_test_data)
sampled_test_data.head()

In [ ]:
sampled_train_data = sample_ts_data(new_df)
sampled_train_data.head()

In [ ]:
spx.bands

In [ ]:
spx.indices

In [ ]:
spx.indices['S2REP']

In [ ]:
new_train_indices = spx.computeIndex(index=['BI',"CSI","AVI","SAVI","EVI","IRECI","MCARI","GNDVI","S2REP"],
                               params={"R" : sampled_train_data['RED'],"B" : sampled_train_data['Blue'],
                                       "G" : sampled_train_data['Green'],"N" : sampled_train_data['NIR'],
                                       "g":2.5,"C1":6.0, "C2":7.5,
                                       "RE1" : sampled_train_data['Red_Edge'], "RE2" : sampled_train_data['Red_Edge_2'],
                                       "RE3" : sampled_train_data['Red_Edge_3'],"L" : 0.5,
                                       "S1" : sampled_train_data['SWIR1'],"S2" : sampled_train_data['SWIR2']})
new_test_indices = spx.computeIndex(index=['BI',"CSI","AVI","SAVI","EVI","IRECI","GNDVI","MCARI","S2REP"],
                               params={"R" : sampled_test_data['RED'],"B" : sampled_test_data['Blue'],
                                       "G" : sampled_test_data['Green'],"N" : sampled_test_data['NIR'],
                                       "g":2.5,"C1":6.0, "C2":7.5,
                                       "RE1" : sampled_test_data['Red_Edge'], "RE2" : sampled_test_data['Red_Edge_2'],
                                       "RE3" : sampled_test_data['Red_Edge_3'],"L" : 0.5,
                                       "S1" : sampled_test_data['SWIR1'],"S2" : sampled_test_data['SWIR2']})


In [ ]:
full_train_data = pd.concat([sampled_train_data, new_train_indices], axis=1)
full_test_data = pd.concat([sampled_test_data, new_test_indices], axis=1)

In [ ]:
full_train_data.head()

In [ ]:
# Ensure the date indices are in datetime format
date_indices = pd.to_datetime(full_train_data['time'], errors='coerce')

# Find the minimum and maximum dates
date_range = (date_indices.min(), date_indices.max())
print(f"Date range: {date_range}")

In [ ]:
id_to_labels = {0 : "forest", 1 : "cocoa", 2 : "palm"}
labels_to_id = dict([(value, key) for key,value in id_to_labels.items()])

In [ ]:
# Create a color palette for plotting
colors = ["#E33F62","#3FDDE3","#4CBA4B"]

derived_train_indices = new_train_indices.copy()
derived_train_indices['vegetation_cover'] = sampled_train_data.Target.astype(int).map(id_to_labels)

sampled_data_to_plot = derived_train_indices.sample(n=30)

# Plots a pairplot to check the behaviour of derived indices
plt.figure(figsize=(15,20))
grid = sns.PairGrid(sampled_data_to_plot, hue="vegetation_cover",palette=sns.color_palette(colors))
grid.map_lower(sns.scatterplot)
grid.map_upper(sns.kdeplot, fill=True, alpha=.55)
grid.map_diag(sns.kdeplot, fill=True)
grid.add_legend()
plt.show()


Classification using random forest

In [ ]:
def resample_data(data : pd.DataFrame, time_column='time',group_key='ID') -> pd.DataFrame:
    data_grouped = data.groupby(by=group_key)
    dfs = []
    for group in data_grouped:
        resampled_group_data = data_grouped.get_group(group[0]).resample('10D', 
                                                                         on=time_column).agg('last').reset_index(drop=True)
        dfs.append(resampled_group_data)
    new_data =  pd.concat(dfs)
    return new_data.assign(timestep=(new_data.groupby(by=group_key).cumcount() + 1).map(lambda x : f"t{str(x).zfill(2)}"))
        

In [ ]:
resampled_data = resample_data(full_train_data)
resampled_data.head()

In [ ]:
target_values = resampled_data[['ID','Target']].groupby(by="ID").agg({"Target" : "last"})
target_values['Target'] = target_values.Target.astype(int)
target_values

In [ ]:
id_time_index_data = resampled_data.set_index(keys=['ID','timestep'])
id_time_index_data.head()

In [ ]:
feat_columns = list(set(resampled_data.columns).difference(set(['ID','time','timestep','Target'])))

In [ ]:
t = resampled_data.copy().drop(columns=['Target'])
t_grouped = t.pivot(index='ID',columns='timestep',values=feat_columns)
t_grouped.columns = [f"{col[0]}_{col[1]}" for col in t_grouped.columns]
t_grouped.head(10)

In [ ]:
merged_train_data = t_grouped.merge(target_values, on=['ID'])
merged_train_data.head()

In [ ]:
merged_train_data.Target.value_counts()

In [ ]:
averages_by_target = merged_train_data.groupby('Target').mean()
averages_by_target

In [ ]:
%%script echo skipping

# Define the objective function for Optuna
def objective(trial):
    # Define hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 5, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    # Create the Random Forest model with the suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform stratified k-fold cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, t_grouped, target_values['Target'], cv=skf, scoring=make_scorer(accuracy_score))

    # Return the mean accuracy
    return scores.mean()

# Create an Optuna study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Print the best hyperparameters
print("Best hyperparameters:", study.best_params)


In [ ]:
%%script echo skipping
# Train the final model with the best hyperparameters
best_params = study.best_params
best_params


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(t_grouped, target_values['Target'],test_size=0.2, random_state=42, stratify=target_values['Target'])

In [ ]:
t_grouped.columns

In [ ]:
%%script echo skipping
final_model = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    max_features=best_params['max_features'],
    random_state=42
)

# Fit the model on the entire dataset
final_model.fit(X_train, y_train)

In [ ]:
%%script echo skipping
final_model.score(X_test, y_test)

In [ ]:
grouped_test_data = full_test_data.groupby('ID')
new_test_df = pd.DataFrame()
new_dfs = []
for group in grouped_test_data:
    test_chip_data = grouped_test_data.get_group(group[0]).bfill().ffill()
    new_dfs.append(test_chip_data)
new_test_df = pd.concat(new_dfs)
new_test_df.head()

In [ ]:
new_test_df.info()

In [ ]:
new_test_df['time'] = pd.to_datetime(new_test_df['time'], errors='coerce')
resampled_test_data = resample_data(new_test_df)
resampled_test_data.head()

In [ ]:
test_pivoted_data = resampled_test_data.pivot(index='ID',columns='timestep',values=feat_columns)
test_pivoted_data.columns = [f"{col[0]}_{col[1]}" for col in test_pivoted_data.columns]

In [ ]:
test_pivoted_data.head()

In [ ]:
test_pivoted_data.shape

In [ ]:
test_pivoted_data.info()

In [ ]:
%%script echo skipping
test_preds = final_model.predict(test_pivoted_data)


In [ ]:
submission_data = test_pivoted_data.reset_index()[['ID']]


In [ ]:
%%script echo skipping
submission_data['Target'] = test_preds
submission_data.head()

In [ ]:
submissions_dir = REPORTS_DIR / 'zindi_submissions'
submissions_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
from datetime import datetime

In [ ]:
%%script echo skipping

timestamp = datetime.now().strftime("%Y%M%d_%H%M%S")
sub_filename = submissions_dir / f"{timestamp}.csv"
with sub_filename.open(mode="w") as fp:
    submission_data.to_csv(fp, index=False)


Try catboost search or xgboost

In [ ]:
import numpy as np
import torch

In [ ]:
cuda_available = torch.cuda.is_available()

In [ ]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}

# Define the objective function for Optuna
def xgb_objective(trial):
    # Define hyperparameters to tune
    param = {
        'objective': 'multi:softmax',
        'tree_method' : 'gpu_hist' if cuda_available else 'hist',
        'num_class': len(np.unique(y_train)),
        'learning_rate': trial.suggest_float('learning_rate', 0.008, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state' : 42,
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Train the model
    model = xgb.train(param, dtrain, num_boost_round=100, evals=[(dtest, 'eval')], early_stopping_rounds=10,
                      verbose_eval=False)

    # Predict and calculate accuracy
    preds = model.predict(dtest)
    accuracy = accuracy_score(y_test, preds)
    return accuracy

# Create an Optuna study and optimize
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=40)


In [ ]:

# Print the best hyperparameters
print("Best hyperparameters:", xgb_study.best_params)

# Train the final model with the best hyperparameters
xgb_best_params = xgb_study.best_params
xgb_best_params['objective'] = 'multi:softmax'
xgb_best_params['num_class'] = len(np.unique(y_train))
xgb_best_params['random_state'] = 42

xgb_model = xgb.train(xgb_best_params, xgb.DMatrix(X_train, label=y_train), num_boost_round=100)

# Evaluate the final model
final_preds = xgb_model.predict(xgb.DMatrix(X_test))
final_accuracy = accuracy_score(y_test, final_preds)
print(f"Final XGB model accuracy: {final_accuracy}")

In [ ]:
xgb_best_params

In [ ]:
xgb_test_data = xgb.DMatrix(test_pivoted_data)
xgb_test_preds = xgb_model.predict(xgb_test_data)

In [ ]:
xgb_submission_data = submission_data.copy()[['ID']]
xgb_submission_data['Target'] = xgb_test_preds.astype(int)
xgb_submission_data.head()

In [ ]:
timestamp = datetime.now().strftime("%Y%M%d_%H%M%S")
sub_filename = submissions_dir / f"xgb_{timestamp}.csv"
with sub_filename.open(mode="w") as fp:
    xgb_submission_data.to_csv(fp, index=False)